# Token Caching in a LangGraph ReAct Agent

This tutorial uses one validated model and interface—`gpt-5.6-luna` through SAP Generative AI Hub's LangChain Responses API—to make prompt-cache behavior visible inside a real ReAct tool loop:

```text
START -> assistant -> [tools_condition] -> tools -> assistant -> ... -> END
```

**Audience**

- Developers who already know basic LangGraph tool calling and use SAP Generative AI Hub.

**Prerequisites**

- SAP AI Core credentials in the adjacent `.env` file.
- A running `gpt-5.6-luna` deployment in the configured resource group.
- The dependencies from `requirements.txt` installed in the repository `.venv`.


## Outline

1. Understand the cache lifecycle and token accounting.
2. Configure GPT-5.6 Luna through LangChain's Responses API.
3. Build one deterministic four-step ReAct agent.
4. Inspect raw `AIMessage.usage_metadata` before normalization.
5. Compare per-call and cumulative telemetry.
6. Run a compact explicit-breakpoint comparison.
7. Review trade-offs and best practices

## What prompt caching changes—and what it does not

Prompt caching reuses model-side computation for an unchanged prompt prefix. It does not remove that prefix from the logical request.

| Event | What the next response can report |
|---|---|
| **Cold write** | No matching prefix exists, so eligible prefix tokens can be written to cache. |
| **Warm read** | A later request matches the prefix and reads cached tokens. |
| **Incremental write** | The old prefix is read while the newly appended suffix is written for later reuse. |
| **Cache miss** | No eligible match is reported; input is processed without a cache read. |

Prompt caching is also not LangGraph state, checkpoint persistence, conversation memory, or `previous_response_id`. Those features decide where conversation data lives or how it is referenced. Prompt caching decides whether repeated input computation is reused.

This notebook uses a fresh run ID inside an otherwise stable system prompt. That prevents an older run from warming call 1 while keeping the prefix byte-identical within the active run.

## Reading raw token telemetry

The LangChain model call returns an `AIMessage`. For the GPT-5.6 Luna Responses shape validated for this tutorial, `message.usage_metadata` looks like:

```python
{
    "input_tokens": ...,
    "output_tokens": ...,
    "total_tokens": ...,
    "input_token_details": {
        "cache_read": ...,
        "cache_creation": ...,
    },
    "output_token_details": {"reasoning": ...},
}
```

| Raw field | Meaning in this observed response shape |
|---|---|
| `input_tokens` | Complete prompt-side token volume, including reported cache reads and writes. |
| `input_token_details.cache_read` | Tokens retrieved from the prompt cache. |
| `input_token_details.cache_creation` | Tokens written to the prompt cache on this call. |
| `output_tokens` | Tokens generated by the model, including any output detail categories reported separately. |
| `total_tokens` | Provider-reported total for the call. |

The raw response does not name uncached input directly. We derive it as:

```text
uncached_input_tokens = input_tokens - cache_read - cache_creation
```

That identity is specific to the field accounting observed here. Other model/API combinations may expose different names or accounting. A missing field means the interface did not report it; a reported zero means it reported no activity.

## 1. Imports and environment

The notebook uses SAP Generative AI Hub's `ChatOpenAI` wrapper because the model runs inside a LangGraph tool loop. The adjacent `normalize_usage.py` keeps the raw input field separate from derived uncached input.

In [1]:
from __future__ import annotations

from uuid import uuid4

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI

from normalize_usage import normalize_usage

load_dotenv(".env", override=True)
print("Environment loaded.")

Environment loaded.


## 2. Configuration

The main run explicitly selects GPT-5.6's implicit cache mode and the Responses API. `STABLE_FACTS` creates a deterministic reusable prefix for the demonstration; in a real agent, that prefix would be system policy, tool schemas, reference documents, or other unchanged context.

Four ordered tool steps produce five model calls—enough to observe a cold write followed by several reuse opportunities without paying for a 27-call provider comparison.

In [2]:
MODEL_NAME = "gpt-5.6-luna"
MAIN_CACHE_MODE = "implicit"
DEMO_STEPS = 4
STABLE_FACTS = 400
OBSERVATION_FACTS = 30

display(
    pd.DataFrame(
        [
            {
                "model": MODEL_NAME,
                "api": "Responses through ChatOpenAI",
                "cache_mode": MAIN_CACHE_MODE,
                "tool_steps": DEMO_STEPS,
                "expected_model_calls": DEMO_STEPS + 1,
            }
        ]
    )
)

,model,api,cache_mode,tool_steps,expected_model_calls
0,gpt-5.6-luna,Responses through ChatOpenAI,implicit,4,5


## 3. Deterministic stable context and tool

The tool returns one ordered observation at a time. Each assistant tool-call message and `ToolMessage` is appended to `MessagesState`, so the logical prompt grows without rewriting the earlier prefix.

In [3]:
def build_stable_reference(fact_count: int) -> str:
    """Return deterministic reference text for the repeated prompt prefix.

    Args:
        fact_count: Positive number of stable facts to generate.

    Returns:
        One space-delimited string containing deterministic facts.

    Raises:
        ValueError: If ``fact_count`` is not positive.
    """

    if fact_count < 1:
        raise ValueError("fact_count must be positive")
    return " ".join(
        f"fact-{index}: this reference value is stable."
        for index in range(1, fact_count + 1)
    )


@tool
def read_demo_step(step: int) -> str:
    """Return one deterministic observation for the requested demo step.

    Args:
        step: Ordered step number from 1 through ``DEMO_STEPS``.

    Returns:
        Synthetic tool evidence plus the required next action.

    Raises:
        ValueError: If ``step`` falls outside the configured demo range.
    """

    if not 1 <= step <= DEMO_STEPS:
        raise ValueError(f"step must be between 1 and {DEMO_STEPS}")
    evidence = " ".join(
        f"observation-{step}-{index}: deterministic tool evidence."
        for index in range(1, OBSERVATION_FACTS + 1)
    )
    next_action = (
        f"Next, call read_demo_step with step {step + 1}."
        if step < DEMO_STEPS
        else "All steps are complete. Return one short final sentence."
    )
    return f"Step {step} completed. {evidence} {next_action}"


TOOLS = [read_demo_step]
print(f"Configured {DEMO_STEPS} ordered tool steps.")

Configured 4 ordered tool steps.


## 4. Configure GPT-5.6 Luna through Responses

`use_responses_api=True` makes the interface explicit. The current SAP wrapper inherits the Chat Completions default `n=1`; clearing it prevents that unsupported argument from being forwarded to the Responses client.

This is an interface-specific compatibility step, not a caching requirement.

In [12]:
def make_model(cache_mode: str) -> ChatOpenAI:
    """Create the configured GPT-5.6 Responses model.

    Args:
        cache_mode: Prompt-cache mode requested from the active model route.

    Returns:
        Configured SAP Generative AI Hub LangChain model.
    """

    model = ChatOpenAI(
        proxy_model_name=MODEL_NAME,
        use_responses_api=True,
        prompt_cache_options={"mode": cache_mode},
        reasoning_effort="medium"
    )
    model.n = None
    return model


implicit_model = make_model(MAIN_CACHE_MODE).bind_tools(TOOLS)
print(f"Configured {MODEL_NAME} with {MAIN_CACHE_MODE} caching.")

Configured gpt-5.6-luna with implicit caching.


## 5. Instrument the ReAct assistant node

Telemetry is captured exactly where the model is invoked. Tool nodes do not call the model, so they do not produce model-usage rows.

The graph keeps both the raw `AIMessage` objects and normalized rows. This lets us inspect the provider output first and use normalization only for comparison and arithmetic.

In [13]:
def build_system_prompt(run_id: str, stable_facts: int) -> str:
    """Return one stable system prompt for the complete agent run.

    Args:
        run_id: Fresh identifier used once to isolate this cache experiment.
        stable_facts: Number of deterministic reference facts to include.

    Returns:
        System instruction string reused byte-for-byte within the run.
    """

    instructions = f"""You are running a controlled ReAct cache demonstration.
Run ID: {run_id}
Call read_demo_step for steps 1 through {DEMO_STEPS}, strictly in order.
Make exactly one tool call per assistant response and wait for its ToolMessage.
Never skip, repeat, combine, or parallelize steps.
After step {DEMO_STEPS}, return one short sentence confirming completion.

Stable reference material follows:
"""
    return instructions + build_stable_reference(stable_facts)


def build_instrumented_agent(*, bust_cache: bool = False) -> tuple[object, list[dict], list]:
    """Compile the ReAct graph and return its telemetry buffers.

    Args:
        bust_cache: Whether to change the system prefix on every model call.

    Returns:
        Compiled graph, normalized usage rows, and raw model responses.
    """

    usage_rows: list[dict] = []
    model_responses: list = []
    stable_prompt = build_system_prompt(str(uuid4()), STABLE_FACTS)

    def assistant(state: MessagesState) -> dict:
        """Call the model and append its response plus usage telemetry.

        Args:
            state: Current append-only LangGraph message state.

        Returns:
            Partial state update containing the new assistant response.
        """

        system_prompt = stable_prompt
        if bust_cache:
            system_prompt += f"\nCache-busting call nonce: {uuid4()}"
        response = implicit_model.invoke(
            [SystemMessage(content=system_prompt), *state["messages"]]
        )
        model_responses.append(response)
        usage = normalize_usage(MODEL_NAME, response)
        usage["call"] = len(usage_rows) + 1
        usage_rows.append(usage)
        return {"messages": [response]}

    graph = StateGraph(MessagesState)
    graph.add_node("assistant", assistant)
    graph.add_node("tools", ToolNode(tools=TOOLS, handle_tool_errors=True))
    graph.add_edge(START, "assistant")
    graph.add_conditional_edges("assistant", tools_condition)
    graph.add_edge("tools", "assistant")
    return graph.compile(), usage_rows, model_responses

## 6. Run the implicit-cache ReAct agent

The validation checks only the deterministic graph trace. Cache reads and writes remain telemetry to interpret rather than assertions that make the notebook fail when the active route behaves differently.

In [14]:
def run_agent(*, bust_cache: bool = False) -> tuple[dict, list[dict], list]:
    """Run the ordered ReAct workflow and validate its structural trace.

    Args:
        bust_cache: Whether to change the system prefix on every model call.

    Returns:
        Final graph state, normalized usage rows, and raw model responses.

    Raises:
        RuntimeError: If the model does not complete the expected ordered trace.
    """

    agent, usage_rows, model_responses = build_instrumented_agent(
        bust_cache=bust_cache
    )
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Execute all {DEMO_STEPS} cache-demonstration steps now."
                )
            ]
        },
        {"recursion_limit": 20},
    )
    tool_messages = [
        message for message in result["messages"] if isinstance(message, ToolMessage)
    ]
    expected_calls = DEMO_STEPS + 1
    if len(tool_messages) != DEMO_STEPS or len(usage_rows) != expected_calls:
        raise RuntimeError(
            "The ordered probe did not complete as expected: "
            f"{len(tool_messages)} tool results and {len(usage_rows)} model calls."
        )
    return result, usage_rows, model_responses


RUN_RESULT, USAGE_ROWS, MODEL_RESPONSES = run_agent()
print(
    f"Completed {DEMO_STEPS} tool steps with "
    f"{len(MODEL_RESPONSES)} model calls."
)

Completed 4 tool steps with 5 model calls.


## 7. Inspect raw `usage_metadata`

The first response is the cold call. The second response is the first opportunity to reuse the prefix written by call 1. Displaying both raw dictionaries prevents normalization from hiding the actual SDK contract.

In [15]:
print("Cold call usage_metadata:")
display(MODEL_RESPONSES[0].usage_metadata)

print("First follow-up usage_metadata:")
display(MODEL_RESPONSES[1].usage_metadata)

Cold call usage_metadata:


{'input_tokens': 4205,
 'output_tokens': 33,
 'total_tokens': 4238,
 'input_token_details': {'cache_creation': 4202, 'cache_read': 0},
 'output_token_details': {'reasoning': 12}}

First follow-up usage_metadata:


{'input_tokens': 4566,
 'output_tokens': 19,
 'total_tokens': 4585,
 'input_token_details': {'cache_creation': 361, 'cache_read': 4202},
 'output_token_details': {'reasoning': 0}}

## 8. Normalize without losing provenance

The display uses `provider_input_tokens` for the raw `usage_metadata["input_tokens"]` value and `uncached_input_tokens` for the derived residual. Cache reads and writes remain separate provider-reported columns.

The three prompt-side shares are telemetry ratios, not price discounts. Current prices can apply different rates to uncached input, cache writes, and cache reads.

In [16]:
DISPLAY_COLUMNS = [
    "call",
    "raw_input_tokens",
    "cache_read_input_tokens",
    "cache_write_input_tokens",
    "uncached_input_tokens",
    "output_tokens",
    "total_tokens",
    "cache_read_share_pct",
    "cache_write_share_pct",
    "uncached_share_pct",
]


def usage_frame(rows: list[dict]) -> pd.DataFrame:
    """Return a display-ready GPT-5.6 cache-telemetry table.

    Args:
        rows: Ordered normalized usage rows from one model run.

    Returns:
        DataFrame containing raw, reported, and derived token fields.

    Raises:
        AssertionError: If reported components do not reconcile to raw input.
    """

    frame = pd.DataFrame(rows).copy()
    frame["raw_input_tokens"] = frame["provider_input_tokens"]
    raw_input = frame["raw_input_tokens"].fillna(0)
    cache_read = frame["cache_read_input_tokens"].fillna(0)
    cache_write = frame["cache_write_input_tokens"].fillna(0)
    uncached = frame["uncached_input_tokens"].fillna(0)

    accounted = cache_read + cache_write + uncached
    assert accounted.astype(int).equals(raw_input.astype(int))

    denominator = raw_input.where(raw_input > 0, 1)
    frame["cache_read_share_pct"] = cache_read.div(denominator).mul(100).round(1)
    frame["cache_write_share_pct"] = cache_write.div(denominator).mul(100).round(1)
    frame["uncached_share_pct"] = uncached.div(denominator).mul(100).round(1)
    return frame[DISPLAY_COLUMNS]


MAIN_USAGE = usage_frame(USAGE_ROWS)
display(MAIN_USAGE)

,call,raw_input_tokens,cache_read_input_tokens,cache_write_input_tokens,uncached_input_tokens,output_tokens,total_tokens,cache_read_share_pct,cache_write_share_pct,uncached_share_pct
0,1,4205,0,4202,3,33,4238,0.0,99.9,0.1
1,2,4566,4202,361,3,19,4585,92.0,7.9,0.1
2,3,4913,4563,347,3,19,4932,92.9,7.1,0.1
3,4,5260,4910,347,3,19,5279,93.3,6.6,0.1
4,5,5607,5257,347,3,14,5621,93.8,6.2,0.1


## 9. Cumulative telemetry

Summing the calls answers three different questions:

- How much prompt-side volume did the agent send logically?
- How much was read from cache or written for future reuse?
- How much input remained uncached after accounting for those operations?

The comparison does not estimate money. A current cost estimate would require current rates in the symbolic form:

```text
prompt cost = uncached × uncached rate + writes × write rate + reads × read rate
```

In [17]:
def summarize_usage(frame: pd.DataFrame) -> pd.DataFrame:
    """Return one cumulative telemetry row for the main agent run.

    Args:
        frame: Per-call usage table returned by ``usage_frame``.

    Returns:
        Single-row DataFrame with cumulative token volumes and shares.
    """

    raw_input = int(frame["raw_input_tokens"].sum())
    cache_read = int(frame["cache_read_input_tokens"].fillna(0).sum())
    cache_write = int(frame["cache_write_input_tokens"].fillna(0).sum())
    uncached = int(frame["uncached_input_tokens"].sum())
    denominator = raw_input or 1
    return pd.DataFrame(
        [
            {
                "model": MODEL_NAME,
                "api": "Responses",
                "cache_mode": MAIN_CACHE_MODE,
                "model_calls": len(frame),
                "raw_input_tokens": raw_input,
                "cache_read_input_tokens": cache_read,
                "cache_write_input_tokens": cache_write,
                "uncached_input_tokens": uncached,
                "output_tokens": int(frame["output_tokens"].sum()),
                "cache_read_share_pct": round(100 * cache_read / denominator, 1),
                "cache_write_share_pct": round(100 * cache_write / denominator, 1),
                "uncached_share_pct": round(100 * uncached / denominator, 1),
            }
        ]
    )


display(summarize_usage(MAIN_USAGE))

,model,api,cache_mode,model_calls,raw_input_tokens,cache_read_input_tokens,cache_write_input_tokens,uncached_input_tokens,output_tokens,cache_read_share_pct,cache_write_share_pct,uncached_share_pct
0,gpt-5.6-luna,Responses,implicit,5,24551,18932,5604,15,104,77.1,22.8,0.1


## 10. Explicit system breakpoint comparison

GPT-5.6 also exposes explicit cache breakpoints on supported content blocks. Explicit mode disables the automatic breakpoint and uses the boundaries supplied by the request.

This experiment marks only the stable system content, then runs four turns that append growing conversation content **without adding new breakpoints**:

1. Call 1 creates the cache entry for the marked system prefix.
2. Calls 2, 3, and 4 read that same system prefix, but everything appended after it (prior assistant replies plus each new user turn) is outside the one breakpoint.

Watch the telemetry: `cache_read_input_tokens` stays roughly flat at the size of the system prefix across calls 2 through 4, while `raw_input_tokens` keeps climbing as the transcript grows. That growing gap is the appended content being processed raw on every call. Without manually placing a fresh breakpoint on the latest reusable message, the model does not extend the cache past the original system boundary.

An agent that wants to cache a growing explicit transcript would place a supported breakpoint on the latest message intended for reuse and verify the resulting telemetry.

In [20]:
def run_explicit_comparison() -> tuple[list, list[dict]]:
    """Run four Responses calls with one explicit system breakpoint.

    Only the stable system content carries a breakpoint. The four turns append
    growing conversation content without adding new breakpoints, so later turns
    can reuse the cached system prefix while the appended suffix stays raw.

    Returns:
        Raw model responses and normalized usage rows for all four calls.
    """

    run_id = str(uuid4())
    cache_key = f"token-cache-tutorial-{run_id}"
    stable_prompt = (
        f"Explicit cache comparison run {run_id}. "
        + build_stable_reference(STABLE_FACTS)
    )
    model = make_model("explicit")
    messages: list = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": stable_prompt,
                    "prompt_cache_breakpoint": {"mode": "explicit"},
                }
            ],
        },
        {"role": "user", "content": "Turn 1: reply OK."},
    ]

    responses: list = []
    rows: list[dict] = []
    for call in range(1, 5):
        response = model.invoke(messages, prompt_cache_key=cache_key)
        responses.append(response)
        usage = normalize_usage(MODEL_NAME, response)
        usage["call"] = call
        rows.append(usage)
        if call < 4:
            messages.extend(
                [response, {"role": "user", "content": f"Turn {call + 1}: reply OK."}]
            )
    return responses, rows


EXPLICIT_RESPONSES, EXPLICIT_ROWS = run_explicit_comparison()
for index, response in enumerate(EXPLICIT_RESPONSES, start=1):
    print(f"Explicit call {index} usage_metadata:")
    display(response.usage_metadata)
display(usage_frame(EXPLICIT_ROWS))

Explicit call 1 usage_metadata:


{'input_tokens': 4042,
 'output_tokens': 5,
 'total_tokens': 4047,
 'input_token_details': {'cache_creation': 4028, 'cache_read': 0},
 'output_token_details': {'reasoning': 0}}

Explicit call 2 usage_metadata:


{'input_tokens': 4060,
 'output_tokens': 16,
 'total_tokens': 4076,
 'input_token_details': {'cache_creation': 4028, 'cache_read': 0},
 'output_token_details': {'reasoning': 9}}

Explicit call 3 usage_metadata:


{'input_tokens': 4089,
 'output_tokens': 5,
 'total_tokens': 4094,
 'input_token_details': {'cache_creation': 4028, 'cache_read': 0},
 'output_token_details': {'reasoning': 0}}

Explicit call 4 usage_metadata:


{'input_tokens': 4107,
 'output_tokens': 5,
 'total_tokens': 4112,
 'input_token_details': {'cache_creation': 4028, 'cache_read': 0},
 'output_token_details': {'reasoning': 0}}

,call,raw_input_tokens,cache_read_input_tokens,cache_write_input_tokens,uncached_input_tokens,output_tokens,total_tokens,cache_read_share_pct,cache_write_share_pct,uncached_share_pct
0,1,4042,0,4028,14,5,4047,0.0,99.7,0.3
1,2,4060,0,4028,32,16,4076,0.0,99.2,0.8
2,3,4089,0,4028,61,5,4094,0.0,98.5,1.5
3,4,4107,0,4028,79,5,4112,0.0,98.1,1.9


Notice how the `uncached_input_tokens` values grow over time if we don't create new manual cache compare to the implicit mode.

## How to interpret the two experiments

- In the implicit ReAct run, look for a cold write followed by calls that read the previous prefix and write only the newly appended extension.
- In the explicit comparison, the system-only breakpoint isolates one stable boundary. Later transcript growth remains uncached unless another explicit breakpoint covers it.
- A cache read on call 1 usually means an identical prefix was already warm; the per-run nonce is intended to prevent that here.
- A single nonzero field is not enough for a general claim. Interpret the ordered sequence and record the model, API, SDK, route, tenant, and date.
- Raw totals are not a model-quality comparison and cache shares are not price discounts.

## When caching can backfire

- **Writes need future reuse.** Writing a prefix has value only when later requests read it under favorable current pricing.
- **The prefix is fragile.** Changing content before a cache boundary can cause a miss or a new write. Timestamps, per-call nonces, reordered tools, and edited earlier turns are common causes.
- **Long pauses matter.** Cache retention is model/API specific. Recheck current documentation before relying on a cache across slow tools or human approval pauses.
- **Small or changing prompts may not benefit.** Eligibility and thresholds can vary. Measure the active route instead of hard-coding a universal number.
- **Explicit control adds responsibility.** A poorly placed breakpoint can write content that is never reused or leave useful transcript growth uncached.

## Best practices

1. Inspect raw `usage_metadata` before normalizing or aggregating it.
2. Distinguish missing fields from provider-reported zeros.
3. Keep stable system context, tool schemas, and references before changing content.
4. Keep the measured transcript append-only; evaluate trimming or summarization separately.
5. Record telemetry at the exact `invoke` or `ainvoke` boundary.
6. Use a cold-run nonce and inspect several consecutive calls.
7. Treat implicit and explicit modes as interface capabilities to test—not provider-wide identities.
8. Re-run the probe after model, API, SDK, route, or tenant changes.
9. Keep pricing, TTL, and threshold statements conditional on current documentation.